In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image

W0, H0 = 1296, 966
CROP_EACH_SIDE = 165
OUT_WH = (350, 350)

x0 = CROP_EACH_SIDE
x1 = W0 - CROP_EACH_SIDE
crop_w = x1 - x0
assert crop_w == H0 == 966, (crop_w, H0)

_RESIZE = Image.Resampling.LANCZOS
print(f" (left, upper, right, lower) = ({x0}, 0, {x1}, {H0}) → {crop_w}×{H0}")
print(f": {OUT_WH[0]}×{OUT_WH[1]}")

In [ ]:
def find_repo_with_sugar_img(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        img_dir = p / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img"
        if img_dir.is_dir():
            return p
    raise FileNotFoundError(
        " EWS/data/sugar-beets-2016-DatasetNinja/ds/img，。"
    )


REPO = find_repo_with_sugar_img(Path.cwd())
IMG_DIR = REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img"
OUT_DIR = REPO / "EWS" / "data" / "sugar-beets-2016-DatasetNinja" / "ds" / "img_350_crop"


OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("IMG_DIR:", IMG_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def process_one(src: Path, dst: Path) -> None:
    im = Image.open(src).convert("RGB")
    if im.size != (W0, H0):
        raise ValueError(f" {W0}×{H0}， {im.size[0]}×{im.size[1]}: {src.name}")
    cropped = im.crop((x0, 0, x1, H0))
    if cropped.size != (crop_w, H0):
        raise RuntimeError(f": {cropped.size}")
    out = cropped.resize(OUT_WH, _RESIZE)
    out.save(dst, format="PNG", optimize=True)


paths = sorted(IMG_DIR.glob("rgb_*.png"))
if not paths:
    raise FileNotFoundError(f" rgb_*.png: {IMG_DIR}")

print(f": {len(paths)} ")

In [ ]:
from tqdm import tqdm

for p in tqdm(paths, desc="crop+resize"):
    process_one(p, OUT_DIR / p.name)

print(f"，: {OUT_DIR}")

In [ ]:
sample = OUT_DIR / paths[0].name
chk = Image.open(sample)
arr = np.asarray(chk)
print(sample.name, "size", chk.size, "dtype", arr.dtype, "shape", arr.shape)